# Data Cleaning for UNSW-NB15 dataset

In [ ]:
from google.colab import userdata
token = userdata.get('GIT_TOKEN')
username = userdata.get('USERNAME')
email = userdata.get('EMAIL')
!git config --global user.email {email}
!git config --global user.name {username}
repo = "cil-intrusion-detection"

In [ ]:
!git clone https://{token}@github.com/{username}/{repo}
%cd {repo}
!git config pull.rebase false # if you need to do some merging
!git pull origin main

## 1. Importing the dataset

In [ ]:
import pandas as pd
df = pd.read_csv('data/raw/UNSW_NB15_training-set.csv')
df_test = pd.read_csv('data/raw/UNSW_NB15_testing-set.csv')

In [ ]:
print("Train shape:", df.shape)
print("Test shape:", df_test.shape)

display(df.head())

id column should be dropped as it's redundant

In [ ]:
df.describe()

In [ ]:
# Remove duplicates and drop unnecessary columns
df = df.drop_duplicates()
df_test = df_test.drop_duplicates()
print("Train shape:", df.shape)
print("Test shape:", df_test.shape)

In [ ]:
print(df['label'].value_counts())
print(df['attack_cat'].value_counts())

We can see that `label` column contains only two labels, 0 and 1 for binary classification of attack with $0 = benign$ and $1 = attack$. The `attack_cat`, instead contains labels for multi-class classification referring to 9 different type of attacks. However, `Worms` attack were removed due to the ex-
tremely small number of samples (130 instances), which is insufficient for stable incremental training.

In [ ]:
df = df.drop(columns=['id', 'label', 'is_sm_ips_ports'], errors='ignore')
df_test = df_test.drop(columns=['id', 'label', 'is_sm_ips_ports'], errors='ignore')

In [ ]:
# Filter out rows with 'Worms' attack category
df = df[df['attack_cat'] != 'Worms'] 
df_test = df_test[df_test['attack_cat'] != 'Worms']  

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols   = df_test.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Categorical columns:", categorical_cols)

There are other three categorical columns in the dataset: `proto`, `service` and `state`.

In [ ]:
print(df['service'].value_counts())
print(df['state'].value_counts())
print(df['proto'].value_counts())

As these columns contain `string` values we used one-hot-econding

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Categorical variables
cat_vars = ['proto', 'service', 'state']

# Initialize OneHotEncoder
encoder = OneHotEncoder(
    sparse_output=False,      # Returns dense array (better for pandas/NN)
    handle_unknown='ignore'   # Avoid crash if a new category appears
)

# Apply One-Hot Encoding
encoded_array = encoder.fit_transform(df[cat_vars])
encoded_array_test = encoder.transform(df_test[cat_vars])

# Get encoded column names
encoded_cols = encoder.get_feature_names_out(cat_vars)

# Create DataFrame for encoded values
df_encoded = pd.DataFrame(encoded_array, columns=encoded_cols, index=df.index)
df_encoded_test = pd.DataFrame(encoded_array_test, columns=encoded_cols, index=df_test.index)

# Concatenate encoded columns and drop original categorical columns
df = pd.concat([df.drop(columns=cat_vars), df_encoded], axis=1)
df_test = pd.concat([df_test.drop(columns=cat_vars), df_encoded_test], axis=1)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

counts = df["attack_cat"].value_counts()
threshold = 0.01  # 1%

# Convert to proportions
proportions = counts / counts.sum()

# Group small categories
large = counts[proportions >= threshold]
small = counts[proportions < threshold]

if not small.empty:
    large["Others"] = small.sum()

plt.figure(figsize=(6,6))
wedges, texts, autotexts = plt.pie(
    large,
    labels=large.index,
    autopct='%1.1f%%',
)

for text in texts:
    text.set_rotation(45)
    text.set_ha('right')

plt.title('Distribution of attack categories')
plt.ylabel('')
plt.show()

In [ ]:
import zipfile
from pathlib import Path
import tempfile
import shutil

#TODO check the file is saved in the correct repository folder

def export_zip_with_root(df, df_test, zip_path):
    zip_path = Path(zip_path)
    zip_path.parent.mkdir(parents=True, exist_ok=True)  # Create directory if not exists

    temp_dir = Path(tempfile.mkdtemp())  # Temporary directory to hold CSVs

    # ROOT "2015" (required for loader structure)
    root = temp_dir / "2015"
    train_dir = root / "train"
    test_dir = root / "test"

    train_dir.mkdir(parents=True, exist_ok=True)  # Create train directory
    test_dir.mkdir(parents=True, exist_ok=True)   # Create test directory

    # Export TRAIN CSV per class
    for attack in sorted(df['attack_cat'].unique()):
        subset = df[df['attack_cat'] == attack]
        safe_name = str(attack).strip().replace(" ", "_").replace("/", "_")
        subset.to_csv(train_dir / f"{safe_name}.csv", index=False)

    # Export TEST CSV per class
    for attack in sorted(df_test['attack_cat'].unique()):
        subset = df_test[df_test['attack_cat'] == attack]
        safe_name = str(attack).strip().replace(" ", "_").replace("/", "_")
        subset.to_csv(test_dir / f"{safe_name}.csv", index=False)

    # Create ZIP preserving "2015/" as root
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for file in temp_dir.rglob("*.csv"):
            arcname = file.relative_to(temp_dir)  # Keep "2015/train/..." structure
            z.write(file, arcname)

    shutil.rmtree(temp_dir)  # Clean up temporary files
    print(f"ZIP compatible with loader created at: {zip_path}")

In [ ]:
export_zip_with_root(df, df_test, "2015.zip")